# Sistema A – Método de Gauss-Jacobi

$$
\begin{cases}
2x_1 - x_2 + x_3 = -1\\
2x_1 + 2x_2 + 2x_3 = 4\\
-x_1 - x_2 + 2x_3 = -5
\end{cases}
$$

$$
X^{(0)} = \begin{pmatrix}0\\0\\0\end{pmatrix}, \qquad E_R < 0{,}00001, \qquad \text{limite de 100 iterações}
$$

In [ ]:
import numpy as np

## Sistema e parâmetros

In [ ]:
A = [[2, -1, 1],
     [2, 2, 2],
     [-1, -1, 2]]
b = [-1, 4, -5]

x0 = [0, 0, 0]
epsilon = 0.00001
max_iterations = 100

## Critério das linhas

$$\alpha_k = \frac{\sum_{j \neq k} |a_{kj}|}{|a_{kk}|}, \qquad \alpha = \max_k \alpha_k < 1$$

In [ ]:
def row_criterion(A):
    n = len(A)
    alphas = []
    for k in range(n):
        soma = sum(abs(A[k][j]) for j in range(n) if j != k)
        alphas.append(soma / abs(A[k][k]))
    return alphas, max(alphas) < 1


alphas, satisfied = row_criterion(A)
print("alphas:", alphas)
print("critério das linhas:", "satisfeito" if satisfied else "NÃO satisfeito")

## Matriz de iteração

$X^{(k+1)} = F X^{(k)} + D$, com $F_{ij} = -a_{ij}/a_{ii}$ (para $i \neq j$) e $D_i = b_i / a_{ii}$.



In [ ]:
def iteration_matrix(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    diagonal = np.diag(A)
    F = -A / diagonal[:, None]
    np.fill_diagonal(F, 0)
    D = b / diagonal
    return F, D


F, D = iteration_matrix(A, b)
spectral_radius = max(abs(np.linalg.eigvals(F)))

print("F =\n", F)
print("D =", D)
print("raio espectral de F =", spectral_radius)
print("converge:", spectral_radius < 1)

## Gauss-Jacobi

In [ ]:
def relative_error(x_new, x_old):
    norm_new = np.max(np.abs(x_new))
    if norm_new == 0:
        return 0.0
    return np.max(np.abs(x_new - x_old)) / norm_new


def jacobi(A, b, x0, epsilon, max_iterations=100, verbose=False):
    n = len(b)
    x = np.array(x0, dtype=float)

    for iterations in range(1, max_iterations + 1):
        x_new = np.zeros(n)
        for i in range(n):
            soma = sum(A[i][j] * x[j] for j in range(n) if j != i)
            x_new[i] = (b[i] - soma) / A[i][i]

        error = relative_error(x_new, x)
        if verbose:
            print(f"{iterations:3} {x_new} E_R = {error:.5f}")

        if error < epsilon:
            return x_new, iterations, True
        x = x_new

    return x, max_iterations, False

Primeiras iterações:

In [ ]:
x, iterations, converged = jacobi(A, b, x0, epsilon, max_iterations=8, verbose=True)

Execução completa (limite de 100 iterações):

In [ ]:
x, iterations, converged = jacobi(A, b, x0, epsilon, max_iterations)

print("iterações:", iterations)
print("convergiu:", converged)
print("x =", x)

Solução exata, para comparação:

In [ ]:
print(np.linalg.solve(np.array(A, dtype=float), np.array(b, dtype=float)))